# Module 9.3: PagedAttention & Continuous Batching

In Module 7.1 (KV Caching), we implemented standard **KV Caching**. It allows us to avoid re-computing Attention values for tokens we've already processed. 

However, when serving an LLM to thousands of users simultaneously (like OpenAI or Anthropic does), standard KV Caching creates a massive problem: **Memory Fragmentation**.

## 1. The Fragmentation Problem

In standard batching, you must pre-allocate contiguous chunks of GPU memory for each user's KV Cache up to their *maximum* sequence length. 

If User A generates 100 tokens, and User B generates 2000 tokens, the GPU memory allocator has to guess the max length. Pre-allocating wastes huge amounts of memory if the output is short. But dynamic allocation leads to fragmentation: tiny holes of free memory scattered across the GPU that are too small to fit a new user's contiguous cache block.

**Result**: Only 20-40% of the GPU VRAM is actually utilized for caching. The rest is wasted.

## 2. The Solution: PagedAttention (vLLM)

### Borrowing from Operating Systems
Decades ago, CPU RAM solved fragmentation using **Virtual Memory and Paging**. 
Instead of finding a contiguous megabyte for a program, the OS splits RAM into strict 4KB "Pages". An application gets a Virtual memory map that points to scattered Physical Pages. 

**PagedAttention** does this for GPU KV Caches:
1. We slice the KV Cache into fixed-size "Blocks" (e.g., each block holds 16 tokens).
2. We maintain a **Block Table** map.
3. When computing Attention, the kernel fetches the blocks piece-by-piece from scattered physical memory locations, instead of requiring one massive contiguous tensor.

In [ ]:
import torch

torch.manual_seed(0)  # Reproducible results

# Constants
BLOCK_SIZE = 4              # Tokens per block (normally 16 or 32)
NUM_BLOCKS = 10            # Total physical blocks available in VRAM
HEAD_DIM = 64             # Size of K/V vectors per token
MAX_BLOCKS_PER_SEQ = 4    # Cap on blocks one sequence may hold (= 16 tokens here)

# 1. The Physical KV Cache Pools (scattered in VRAM)
# Shape: [NUM_BLOCKS, BLOCK_SIZE, HEAD_DIM]
# Real engines cache BOTH keys and values. They are managed identically, so
# everything we do to physical_k_cache applies one-for-one to physical_v_cache.
physical_k_cache = torch.zeros(NUM_BLOCKS, BLOCK_SIZE, HEAD_DIM)
physical_v_cache = torch.zeros(NUM_BLOCKS, BLOCK_SIZE, HEAD_DIM)

# 2. The Block Table (the virtual-to-physical lookup) for a single user request.
# Sized from MAX_BLOCKS_PER_SEQ so the capacity limit is explicit, not a magic
# hard-coded length. -1 means "logical slot not yet mapped to a physical block".
block_table = [-1] * MAX_BLOCKS_PER_SEQ

print(f"Physical K cache shape: {physical_k_cache.shape}")
print(f"Physical V cache shape: {physical_v_cache.shape}")
print(f"Block table (capacity {MAX_BLOCKS_PER_SEQ} blocks "
      f"= {MAX_BLOCKS_PER_SEQ * BLOCK_SIZE} tokens): {block_table}")

## 3. Allocating and Filling Blocks

Let's simulate processing a request of 6 tokens.

In [ ]:
seq_len = 6

# We need enough physical blocks to hold 6 tokens. (6 / 4 = 1 remainder 2 -> 2 blocks)
blocks_needed = (seq_len + BLOCK_SIZE - 1) // BLOCK_SIZE
assert blocks_needed <= MAX_BLOCKS_PER_SEQ, "Sequence exceeds this seq's block capacity!"

# Simulate a fragmented pool: blocks 0, 1, 2, 4 are already "in use" by other people.
# The free pool is the list of physical blocks the allocator can hand out.
free_pool = [3, 5, 6, 7, 8, 9]

# Allocate by popping free physical blocks into our logical slots.
print(f"Sequence length: {seq_len} -> requires {blocks_needed} blocks.")
for i in range(blocks_needed):
    physical_block_idx = free_pool.pop(0)
    block_table[i] = physical_block_idx

print(f"Virtual block table map: {block_table}  (logical slot -> physical block)")
print(f"Free pool now: {free_pool}")

# The K and V vectors we want to store for these 6 tokens.
mock_k_tensors = torch.randn(seq_len, HEAD_DIM)
mock_v_tensors = torch.randn(seq_len, HEAD_DIM)

# Inject each token into its scattered physical page (both K and V).
for i in range(seq_len):
    logical_block_idx = i // BLOCK_SIZE
    offset_in_block   = i % BLOCK_SIZE
    physical_block_idx = block_table[logical_block_idx]

    physical_k_cache[physical_block_idx, offset_in_block] = mock_k_tensors[i]
    physical_v_cache[physical_block_idx, offset_in_block] = mock_v_tensors[i]

print("Tokens (K and V) injected into scattered physical memory pages.")

### The payoff: reading the scattered cache back through the block table

Writing to scattered memory is only useful if we can read it back *as if it were contiguous*. That is exactly what the attention kernel does: it walks the block table, gathers each physical block in logical order, and stitches them into the contiguous K (and V) tensors attention needs. Let's reconstruct and verify it matches the original.

In [ ]:
def gather_contiguous(physical_cache, block_table, seq_len):
    """Reconstruct the logically-contiguous K/V tensor from scattered blocks.

    Walk the block table in logical order, concatenate the physical blocks it
    points to, then trim to the real sequence length (the last block may be
    only partially filled).
    """
    blocks_used = (seq_len + BLOCK_SIZE - 1) // BLOCK_SIZE
    gathered = torch.cat(
        [physical_cache[block_table[i]] for i in range(blocks_used)], dim=0
    )
    return gathered[:seq_len]  # drop padding rows in the final block

reconstructed_k = gather_contiguous(physical_k_cache, block_table, seq_len)
reconstructed_v = gather_contiguous(physical_v_cache, block_table, seq_len)

print(f"Reconstructed K shape: {reconstructed_k.shape} (original: {mock_k_tensors.shape})")

# The whole abstraction is only correct if the scattered round-trip is lossless.
assert torch.equal(reconstructed_k, mock_k_tensors), "K reconstruction mismatch!"
assert torch.equal(reconstructed_v, mock_v_tensors), "V reconstruction mismatch!"
print("PASS: scattered blocks reassembled into the exact original K and V tensors.")
print("The attention kernel can now treat the paged cache as if it were contiguous.")

## 4. Continuous Batching

Because PagedAttention decouples memory allocation from sequence length, we can do **Continuous Batching**.

In older systems, you wait for 4 users to submit requests. You batch them. If User A finishes after 10 tokens, and User D takes 2000 tokens, the GPU slot for User A stays "locked" and idle while waiting for D to finish.

With Continuous Batching, the moment User A's generation finishes, their blocks are returned to the pool, and the server immediately injects User E into the batch at the next iteration. The batch size is dynamic and iterates at the token level.

The original vLLM paper reported up to ~24x higher throughput than vanilla HuggingFace Transformers in their best-case benchmarks (the margin is smaller against already-optimized servers). Treat such figures as illustrative best cases — the real gain depends on the workload and what you compare against. Let's make the "free and reuse blocks" mechanic concrete.

In [ ]:
# Demo: free a finished sequence's blocks, then reuse them for a new request.

# Recall our current state from Section 3:
#   - This sequence (call it User A) holds the blocks in `block_table`.
#   - `free_pool` holds whatever is left.
print(f"User A's block table: {block_table}")
print(f"Free pool before A finishes: {free_pool}\n")

# 1. User A finishes generating. Return its physical blocks to the pool.
freed = [b for b in block_table if b != -1]
free_pool.extend(freed)
block_table_A = [-1] * MAX_BLOCKS_PER_SEQ  # A's table is cleared
print(f"User A finished -> freed blocks {freed} back to the pool.")
print(f"Free pool after freeing: {free_pool}\n")

# 2. A brand-new request (User E) arrives needing 5 tokens (-> 2 blocks).
new_seq_len = 5
new_blocks_needed = (new_seq_len + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table_E = [-1] * MAX_BLOCKS_PER_SEQ
for i in range(new_blocks_needed):
    block_table_E[i] = free_pool.pop(0)  # reuses the physical blocks A just freed

print(f"User E arrives needing {new_blocks_needed} blocks.")
print(f"User E's block table: {block_table_E}  (note: reuses A's freed blocks)")
print(f"Free pool after E allocates: {free_pool}")
print("\nNo defragmentation, no copying — just remapping the table. "
      "That is what keeps the GPU busy across requests.")

### 🏋️ Try it yourself

1. **Growing a sequence.** A real sequence grows one token at a time and only allocates a *new* block when the current one fills up. Write `append_token(block_table, free_pool, current_len, k_vec)` that allocates a fresh physical block from `free_pool` only when `current_len % BLOCK_SIZE == 0`, writes `k_vec` at the right offset, and returns the updated length. Append 10 tokens and watch blocks get pulled from the pool lazily.

2. **Out of memory.** Set `free_pool = [3]` (only one free block) and try to allocate a sequence that needs 3 blocks. Detect the failure gracefully and print an "evict or reject request" message — this is exactly the pressure that makes the block accounting matter in a real server.

In [ ]:
# Starter code for Task 1 — lazily grow a sequence one token at a time.

def append_token(block_table, free_pool, current_len, k_vec):
    logical_block = current_len // BLOCK_SIZE
    offset        = current_len % BLOCK_SIZE
    # Allocate a new physical block only when we cross into a fresh logical block.
    if offset == 0:
        if not free_pool:
            raise MemoryError("Out of physical blocks!")
        block_table[logical_block] = free_pool.pop(0)
    physical = block_table[logical_block]
    physical_k_cache[physical, offset] = k_vec
    return current_len + 1

# Demo: grow a fresh sequence by 10 tokens from a fresh pool.
demo_table = [-1] * MAX_BLOCKS_PER_SEQ   # capacity 4 blocks = 16 tokens
demo_pool  = [0, 1, 2, 3, 4]
length = 0
for t in range(10):
    length = append_token(demo_table, demo_pool, length, torch.randn(HEAD_DIM))
    print(f"After token {length}: block_table={demo_table}, free_pool={demo_pool}")

# TODO (Task 2): set demo_pool = [3] and request 3 blocks; catch MemoryError
#                and print an "evict or reject request" message.

---

## 🎓 Series Capstone: You Built an LLM — Now You Can Serve One

This is the final notebook of the series. Take a moment to see how far you've come, because the pieces now connect into a complete story.

### What you actually built
Earlier in the course you implemented a real Transformer from scratch — tokenization, embeddings, multi-head self-attention, RoPE positional encoding, a KV cache, and the training loop. You didn't just read about it; you trained a tiny model with `scripts/train_tiny.py` and generated text from it with `scripts/generate.py`. That is the entire forward-and-backward engine that powers every model from GPT-2 to Llama 3 — only the scale differs.

### How this module's three techniques stack together
A production serving engine like **vLLM** is not one trick; it is these layers composed on top of the exact architecture you built:

| Technique | Module | Problem it attacks | What it does to *your* model |
|---|---|---|---|
| **Quantization** | 9.1 | Weights are too big; memory-bandwidth bound | Stores your weight matrices in INT8/INT4 so they load ~2x faster and fit in less VRAM |
| **Speculative Decoding** | 9.2 | One slow Target pass per token | Uses a small draft model to propose several tokens, verified in one pass of your big model — with output identical to greedy decoding |
| **PagedAttention + Continuous Batching** | 9.3 | KV cache fragments VRAM; idle GPU slots | Pages the KV cache you built in Module 7.1 into reusable blocks, so many requests share VRAM and the GPU never idles |

Read top to bottom, the flow is: **shrink the weights (quantization) → load them fewer times (speculative decoding) → and while they're loaded, serve as many users as possible (paged attention + continuous batching).** Each technique attacks the *same* root cause from a different angle — the memory-bandwidth bottleneck we opened the module with — and they compose without conflicting.

### The full mental model
```
            ┌─────────────────────────────────────────────┐
            │   Your Transformer (the architecture you      │
            │   built + trained: attention, RoPE, KV cache) │
            └─────────────────────────────────────────────┘
                                  │
        weights stored compactly  ▼   (Module 9.1: Quantization)
            ┌─────────────────────────────────────────────┐
            │   Loaded into VRAM in INT8/INT4               │
            └─────────────────────────────────────────────┘
                                  │
        fewer expensive passes    ▼   (Module 9.2: Speculative Decoding)
            ┌─────────────────────────────────────────────┐
            │   Draft proposes, target verifies in bulk     │
            └─────────────────────────────────────────────┘
                                  │
        many users share VRAM     ▼   (Module 9.3: PagedAttention)
            ┌─────────────────────────────────────────────┐
            │   Paged KV cache + continuous batching        │
            │   = a real serving engine (vLLM-style)        │
            └─────────────────────────────────────────────┘
```

### Where to go next
- **Run a real engine.** Install vLLM or TGI and serve a small open model (e.g. a 1–3B Llama or Qwen). You will recognize every flag: `--quantization`, `--block-size`, `--max-num-seqs`, speculative draft models. They are these notebooks, productionized.
- **Quantize a real checkpoint.** Try `bitsandbytes`, GPTQ, or AWQ on a small Hugging Face model and measure the VRAM and latency change yourself.
- **Scale your own model.** Take `scripts/train_tiny.py`, widen the layers, train on a bigger corpus, and load the result into one of the engines above.
- **Go deeper on correctness.** Implement true speculative *sampling* (the `min(1, p_target/p_draft)` accept/resample rule) so it works for temperature > 0, not just greedy.

You started from "what is a parameter?" and you finish understanding how a state-of-the-art LLM is built *and* served at scale. That is the whole pipeline. Congratulations — you can build and serve an LLM. 🏋️

> **One last stop before Part II:** Module 9.4 (`29_reading_a_real_llm.ipynb`) is your graduation lap — we open the source of real models (nanoGPT, Llama-3) and prove you can now read every line.